# Tile mosaic image acquisitions from Opera Phenix 

This notebook exhibits the original tiling code used to compile Opera Phenix acquisition into contiguous mosaic images for tracking cells across tiles. 

### Original approach

In [1]:
from macrohet import dataio, tile

/home/dayn/analysis/macrohet/macrohet/tile.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
basename = '/mnt/OPERA3/Nathan/data/julio/20251126_Eachan_ProDrug_LIVE__2025-11-26T15_02_56-Measurement 1/'

In [3]:
metadata = dataio.read_harmony_metadata(f'{basename}/Images/Index.idx.xml')

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [4]:
metadata

,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0101K1F1P1R1,Ok,r01c01f01p01-ch1sk1fk1fl1.tiff,1,1,1,1,0,1,1,...,0,0.135310799,0,2025-11-26T15:04:07.02+00:00,405,456,40,1.1,0.1,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
1,0101K1F1P1R2,Ok,r01c01f01p01-ch2sk1fk1fl1.tiff,1,1,1,1,0,2,1,...,0,0.135310799,0,2025-11-26T15:04:07.02+00:00,640,706,40,1.1,0.12,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
2,0101K1F1P1R3,Ok,r01c01f01p01-ch3sk1fk1fl1.tiff,1,1,1,1,0,3,1,...,0,0.135310799,0,2025-11-26T15:04:07.253+00:00,488,522,40,1.1,0.1,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
3,0101K1F1P2R1,Ok,r01c01f01p02-ch1sk1fk1fl1.tiff,1,1,1,2,0,1,1,...,2E-06,0.135312796,0,2025-11-26T15:04:07.503+00:00,405,456,40,1.1,0.1,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
4,0101K1F1P2R2,Ok,r01c01f01p02-ch2sk1fk1fl1.tiff,1,1,1,2,0,2,1,...,2E-06,0.135312796,0,2025-11-26T15:04:07.52+00:00,640,706,40,1.1,0.12,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529195,0512K140F9P9R2,Ok,r05c12f09p09-ch2sk140fk1fl1.tiff,5,12,9,9,139,2,1,...,1.6E-05,0.135129496,166802.44999999998,2025-11-28T13:35:30.303+00:00,640,706,40,1.1,0.12,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
529196,0512K140F9P9R3,Ok,r05c12f09p09-ch3sk140fk1fl1.tiff,5,12,9,9,139,3,1,...,1.6E-05,0.135129496,166802.44999999998,2025-11-28T13:35:30.537+00:00,488,522,40,1.1,0.1,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
529197,0512K140F9P10R1,Ok,r05c12f09p10-ch1sk140fk1fl1.tiff,5,12,9,10,139,1,1,...,1.8E-05,0.135131493,166802.44999999998,2025-11-28T13:35:30.787+00:00,405,456,40,1.1,0.1,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."
529198,0512K140F9P10R2,Ok,r05c12f09p10-ch2sk140fk1fl1.tiff,5,12,9,10,139,2,1,...,1.8E-05,0.135131493,166802.44999999998,2025-11-28T13:35:30.803+00:00,640,706,40,1.1,0.12,"[[0.988676,0,0,-11.3],[0,-0.988676,0,-7.0],[0,..."


In [5]:
metadata[['Row', 'Col']].drop_duplicates()

,Row,Col
0,1,1
270,1,2
540,1,3
810,1,4
1080,1,5
1350,1,11
1620,1,12
1890,5,1
2160,5,2
2430,5,3


In [6]:
image_dir = f'{basename}/Images/'
images = tile.compile_mosaic(image_dir, metadata, row=1, col=1,  n_tile_cols=3, n_tile_rows=3)

In [7]:
images

dask.array<reshape, shape=(140, 3, 10, 3024, 3024), dtype=uint16, chunksize=(1, 1, 1, 1080, 1080), chunktype=numpy.ndarray>

In [18]:
%%time
loaded_images = images.compute().compute()

CPU times: user 11h 22min 57s, sys: 28min 26s, total: 11h 51min 24s
Wall time: 2h 15min 38s


In [13]:
loaded_images.shape

(3, 10, 3024, 3024)

In [8]:
import napari

In [9]:
from tqdm.auto import tqdm

In [18]:
import os
os.makedirs('/mnt/OPERA3/Nathan/data/julio/row1col1_frames/')

In [10]:
viewer = napari.Viewer(title = 'julios data')

In [19]:
metadata['ImageResolutionX'].iloc[0]

'2.9898804047838085E-07'

In [11]:
import napari
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [13]:
# 1. Calculate resolution in microns (SI)
pixel_res_m = float(metadata['ImageResolutionX'].iloc[0])
pixel_res_um = pixel_res_m * 1e6

# 2. Establish the temporal baseline
# We use format='mixed' to handle entries that may or may not have fractional seconds
metadata['AbsTime'] = pd.to_datetime(metadata['AbsTime'], format='mixed')
start_time = metadata['AbsTime'].iloc[0]

# 3. Configure the Viewer HUD
viewer.scale_bar.visible = True
viewer.scale_bar.unit = "um"
viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 12
viewer.text_overlay.position = "top_right"

# 4. Iterate and Capture
for i, frame in tqdm(enumerate(images), total=140):
    viewer.layers.clear()

    # Calculate time delta
    current_time = metadata['AbsTime'].iloc[i]
    delta = current_time - start_time
    
    # Format time as HH:MM:SS
    total_seconds = int(delta.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    time_str = f"{hours:02}:{minutes:02}:{seconds:02}"
    
    viewer.text_overlay.text = f"Time: {time_str}"

    # Add image with scale
    viewer.add_image(
        frame.max(axis=1),
        channel_axis=0,
        colormap=['blue', 'green', 'magenta'], 
        name=['Blue', 'Green', 'Magenta'],
        scale=(pixel_res_um, pixel_res_um),
        blending='additive'
    )

    viewer.screenshot(f'/mnt/OPERA3/Nathan/data/julio/row1col1_frames/frame_{i}.png', canvas_only=True)

  0%|          | 0/140 [00:00<?, ?it/s]

In [14]:
import subprocess
import os 
# Configuration
framerate = 10  # Adjust speed as necessary
input_pattern = '/mnt/OPERA3/Nathan/data/julio/row1col1_frames/frame_%d.png'
output_file = '/mnt/OPERA3/Nathan/data/julio/row1col1_timelapse.mp4'

# The command structure
# -y: Overwrite output files without asking
# -r: Set input frame rate
# -i: Input file pattern (%d is the placeholder for the index)
# -vcodec libx264: Use the H.264 codec (widely compatible)
# -pix_fmt yuv420p: Ensure compatibility with standard media players (QuickTime/Windows Media)
cmd = [
    'ffmpeg',
    '-y',
    '-framerate', str(framerate),
    '-i', input_pattern,
    '-vcodec', 'libx264',
    '-pix_fmt', 'yuv420p',
    output_file
]

# Execute
print(f"Compiling matrix: {output_file}...")
try:
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Construct complete.")
except subprocess.CalledProcessError as e:
    print(f"Glitch in the system: {e.stderr.decode()}")

Compiling matrix: /mnt/OPERA3/Nathan/data/julio/row1col1_timelapse.mp4...
Construct complete.


In [26]:
metadata[metadata['TimepointID']=='2'].iloc[0]['AbsTime']

Timestamp('2025-11-26 15:44:07.443000+0000', tz='UTC')

In [24]:
metadata[metadata['TimepointID']=='1'].iloc[0]['AbsTime']

Timestamp('2025-11-26 15:24:07.740000+0000', tz='UTC')

In [32]:
list(unique_positions.iterrows())

[(0,
  Row    1
  Col    1
  Name: 0, dtype: object),
 (1,
  Row     1
  Col    11
  Name: 1, dtype: object),
 (2,
  Row     1
  Col    12
  Name: 2, dtype: object),
 (3,
  Row    1
  Col    2
  Name: 3, dtype: object),
 (4,
  Row    1
  Col    3
  Name: 4, dtype: object),
 (5,
  Row    1
  Col    4
  Name: 5, dtype: object),
 (6,
  Row    1
  Col    5
  Name: 6, dtype: object),
 (7,
  Row    5
  Col    1
  Name: 7, dtype: object),
 (8,
  Row     5
  Col    11
  Name: 8, dtype: object),
 (9,
  Row     5
  Col    12
  Name: 9, dtype: object),
 (10,
  Row    5
  Col    2
  Name: 10, dtype: object),
 (11,
  Row    5
  Col    3
  Name: 11, dtype: object),
 (12,
  Row    5
  Col    4
  Name: 12, dtype: object),
 (13,
  Row    5
  Col    5
  Name: 13, dtype: object)]

In [31]:
reversed(list(unique_positions.iterrows()))

In [35]:
import os
import subprocess
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import napari

# 1. Configuration and Constants
base_output_dir = '/mnt/OPERA3/Nathan/data/julio/'
framerate = 7  # 140 frames / 20 seconds = 7 fps
minutes_per_frame = 20  # The fixed time interval you requested

# Resolution Calc (SI Units)
pixel_res_m = float(metadata['ImageResolutionX'].iloc[0])
pixel_res_um = pixel_res_m * 1e6

# Identify unique positions
unique_positions = metadata[['Row', 'Col']].drop_duplicates().sort_values(['Row', 'Col']).reset_index(drop=True)

# ---------------------------------------------------------
# PHASE 1: The Rendering (Static Generation)
# ---------------------------------------------------------
print("Phase 1: Generating static frames...")

for idx, pos in tqdm(unique_positions.iterrows(), total=unique_positions.shape[0], desc="Rendering Positions"):
    row_idx = pos['Row']
    col_idx = pos['Col']
    
    # Define and create directory
    save_dir = os.path.join(base_output_dir, f"row{row_idx}col{col_idx}_frames")
    os.makedirs(save_dir, exist_ok=True)

    # Compile Mosaic
    images = tile.compile_mosaic(
        image_dir, 
        metadata, 
        row=row_idx, 
        col=col_idx, 
        n_tile_cols=3, 
        n_tile_rows=3
    )

    # Frame Loop
    for i, frame in enumerate(images):
        viewer.layers.clear()
        
        # --- FIXED TIMESTAMP LOGIC ---
        # We simply multiply the frame index by 20 minutes
        total_minutes = i * minutes_per_frame
        
        # Calculate hours and minutes for the display
        # We don't need seconds since we are jumping by 20 mins
        hours, minutes = divmod(total_minutes, 60)
        time_str = f"{hours:02}:{minutes:02}:00"
        # -----------------------------
        
        # Apply Overlays
        viewer.text_overlay.visible = True
        viewer.text_overlay.text = f"Time: {time_str}"
        viewer.scale_bar.visible = True
        viewer.scale_bar.unit = "um"
        
        # Add Image
        viewer.add_image(
            frame.max(axis=1),
            channel_axis=0,
            colormap=['blue', 'green', 'magenta'], 
            scale=(pixel_res_um, pixel_res_um),
            blending='additive'
        )
        
        # Capture
        viewer.screenshot(f'{save_dir}/frame_{i}.png', canvas_only=True)

# ---------------------------------------------------------
# PHASE 2: The Encoding (Video Compilation)
# ---------------------------------------------------------
print("Phase 2: Compiling video streams...")

for idx, pos in tqdm(unique_positions.iterrows(), total=unique_positions.shape[0], desc="Encoding MP4s"):
    row_idx = pos['Row']
    col_idx = pos['Col']
    
    # Define paths
    save_dir = os.path.join(base_output_dir, f"row{row_idx}col{col_idx}_frames")
    video_output = os.path.join(base_output_dir, f"row{row_idx}col{col_idx}_timelapse.mp4")
    input_pattern = os.path.join(save_dir, 'frame_%d.png')
    
    # FFmpeg Command
    cmd = [
        'ffmpeg',
        '-y',                      # Overwrite
        '-framerate', str(framerate),
        '-i', input_pattern,
        '-vcodec', 'libx264',
        '-pix_fmt', 'yuv420p',     # Compatibility format
        video_output
    ]
    
    try:
        # Execute silently
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except subprocess.CalledProcessError as e:
        print(f"Glitch detected in Row {row_idx} Col {col_idx}: {e.stderr.decode()}")

print("System processing complete.")

Phase 1: Generating static frames...


Rendering Positions:   0%|          | 0/14 [00:00<?, ?it/s]

Phase 2: Compiling video streams...


Encoding MP4s:   0%|          | 0/14 [00:00<?, ?it/s]

System processing complete.


In [ ]:
print("Phase 2: Compiling video streams...")

for idx, pos in tqdm(unique_positions.iterrows(), total=unique_positions.shape[0], desc="Encoding MP4s"):
    row_idx = pos['Row']
    col_idx = pos['Col']
    
    # Define paths
    save_dir = os.path.join(base_output_dir, f"row{row_idx}col{col_idx}_frames")
    video_output = os.path.join(base_output_dir, f"row{row_idx}col{col_idx}_timelapse.mp4")
    input_pattern = os.path.join(save_dir, 'frame_%d.png')
    
    # FFmpeg Command
    cmd = [
        'ffmpeg',
        '-y',                      # Overwrite
        '-framerate', str(framerate),
        '-i', input_pattern,
        '-vcodec', 'libx264',
        '-pix_fmt', 'yuv420p',     # Compatibility format
        video_output
    ]
    
    try:
        # Execute silently
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except subprocess.CalledProcessError as e:
        print(f"Glitch detected in Row {row_idx} Col {col_idx}: {e.stderr.decode()}")

print("System processing complete.")

In [6]:
loaded_images = images.compute().compute() # why do i need to compute twice?

In [7]:
loaded_images.shape

(75, 2, 3, 1198, 1198)

In [8]:
loaded_images

array([[[[[ 107,  107,  106, ...,  109,  107,  106],
          [ 106,  108,  106, ...,  109,  106,  105],
          [ 104,  107,  108, ...,  120,  111,  105],
          ...,
          [ 689,  551,  598, ...,  386,  367,  333],
          [ 575,  402,  413, ...,  399,  375,  335],
          [ 497,  333,  358, ...,  397,  372,  365]],

         [[ 106,  110,  109, ...,  111,  111,  109],
          [ 107,  109,  108, ...,  110,  112,  110],
          [ 109,  111,  110, ...,  115,  113,  111],
          ...,
          [1012,  837,  827, ...,  351,  320,  280],
          [1072,  969,  991, ...,  351,  323,  283],
          [1237, 1031, 1020, ...,  356,  334,  341]],

         [[ 107,  108,  111, ...,  107,  110,  110],
          [ 108,  109,  111, ...,  108,  108,  108],
          [ 108,  111,  110, ...,  110,  109,  109],
          ...,
          [ 567,  679,  922, ...,  146,  146,  142],
          [ 497,  791, 1135, ...,  148,  144,  142],
          [ 442,  547, 1023, ...,  150,  144,  146

In [9]:
import napari

In [ ]:
viewer = napari.Viewer(title = 'Fully loaded mosaic compilation')
viewer.add_image(loaded_images, channel_axis =1, colormap=['green', 'magenta'])

## Optional: Save out fully compiled mosaic image array:

### 1. As Zarr

In [67]:
import numpy as np
import zarr

# Open root Zarr store
z = zarr.open('../data/example_data.zarr', mode='w')
img_grp = z.create_group('images')             # Create 'images/' group
img_grp.create_dataset('0',                    # Create dataset at 'images/0'
    data=loaded_images,
    chunks=(1, 1, 1, 256, 256),
    compressor=None
)


<zarr.core.Array '/images/0' (75, 2, 3, 1198, 1198) uint16>

##### Append some metadata

In [68]:
# Calculate pixel sizes
res_x = float(metadata['ImageResolutionX'].astype(float).iloc[0])
res_y = float(metadata['ImageResolutionY'].astype(float).iloc[0])
z_vals = metadata['PositionZ'].astype(float).unique()
res_z = float(np.ptp(z_vals) / (len(z_vals) - 1)) if len(z_vals) > 1 else 1.0

# Attach metadata to root
z.attrs.update({
    "multiscales": [{
        "version": "0.4",
        "axes": [{"name": a, "type": "time" if a == "t" else "space"} for a in "tczyx"],
        "datasets": [{"path": "images/0", "coordinateTransformations": [{
            "type": "scale", "scale": [1.0, 1.0, res_z, res_y, res_x]
        }]}]
    }],
    "description": "Example time-lapse of Mtb-infected macrophages",
    "axes": "TCZYX",
    "unit": "micrometer",
    "pixel_size": {"X": res_x, "Y": res_y, "Z": res_z},
    "n_timepoints": loaded_images.shape[0],
    "n_channels": loaded_images.shape[1],
    "n_zplanes": loaded_images.shape[2],
    "harmony_metadata": {
        "Channels": metadata['ChannelName'].unique().tolist(),
        "ObjectiveNA": float(metadata['ObjectiveNA'].astype(float).max()),
        "ExposureTime (ms)": float(metadata['ExposureTime'].astype(float).mean())
    }
})


### 2. As TIFF

In [ ]:
import tifffile as tiff

# Save to file
tiff.imwrite(
    '../data/example_data.ome.tiff',
    loaded_images,  # shape (T, C, Z, Y, X)
    photometric='minisblack',
    metadata={'axes': 'TCZYX'},
    bigtiff=True 
)